# 教程 4：三种循环 —— 层循环、步循环、会话循环

一个 agent 和一次问答的区别，在于「再来一遍」。生成的答案不满意，就带着反馈重来；
这一步学到的教训，下一步就该用上；上一个会话攒下的东西，下一个会话不该从零开始。

问题是：「再来一遍」发生在哪个作用域？PLLM 不替你把控制流写死，而是给你三种
粒度不同的循环机制，各自用签名声明：

| 循环 | 一句话 | 机制 |
|------|--------|------|
| 层循环 | 更新发布后，同一 decode 步内把 5–8 层**重跑一遍** | `layer_route` + `layers.apply_updates()` |
| 步循环 | 每一步踩在上一步的终态上 | `Recurrence`（上一步源层 hidden 注入目标层） |
| 会话循环 | 新请求带着上一请求的终态重新生成 | route 共享 lineage + `pllm.carry` 写 carry |

三者有一个共同点，也是本篇的心智主线：**循环不靠 Python 侧的 while/for 驱动，
而是声明进签名，由引擎在正确的时机执行**。你只描述「什么状态流向哪里」，
时机、KV、版本纪律都由签名保证。下面各做一个，做完你就能把三种作用域自由组合。

动手之前，先跑一个不带任何 workload 的基线文本作为「原点」——它会是一句平平无奇
的续写。记住它，后面每个循环都会让生成轨迹偏离这条基线，而那正是循环生效的证据。

In [1]:
import gc
from dataclasses import dataclass
from typing import Annotated

import torch
import pllm
from pllm import UpdateOutput
from vllm import SamplingParams
from pllm.config import ModelConfig
from pllm.contracts.parameter_overlay import ParameterModuleKind, ParameterRole

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
PROMPT = "The sky is"

def config(max_num_seqs=1):
    # 层路由在 vLLM 的 CUDA graph 之外重跑层，所以关闭 graph。
    return ModelConfig(MODEL, "bfloat16", 64, max_num_seqs, 64, 0.2, seed=7)

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()

def text_of(handle, prompt, **kwargs):
    return handle.generate([prompt], SamplingParams(max_tokens=16,
        temperature=0.0, ignore_eos=True), use_tqdm=False, **kwargs)[0].outputs[0].text

plain = pllm.make_llm(config=config())
baseline = text_of(plain, PROMPT)
plain.llm_engine.engine_core.shutdown()
del plain
cleanup()
print("baseline:", baseline)

/workspace/pllm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-10 18:42:40,918	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 09-10 18:42:41 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 64, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 64, 'max_num_seqs': 1, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


INFO 09-10 18:42:43 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 18:42:43 [model.py:1883] Using max model len 64


WARNING 09-10 18:42:43 [arg_utils.py:2620] This model does not officially support disabling chunked prefill. Disabling this manually may cause the engine to crash or produce incorrect outputs.


... (66 outputs left out)

INFO 09-10 18:42:56 [core_client.py:690] [shutdown] MPClient: engine manager stopped


INFO 09-10 18:42:56 [core_client.py:691] [shutdown] MPClient: cleaning up background resources


INFO 09-10 18:42:56 [core_client.py:693] [shutdown] MPClient: complete


baseline:  a beautiful place. It is a place of endless possibilities. The sky is a


## 步循环：上一步的终态注入这一步

层循环发生在「一步之内」；步循环发生在「步与步之间」：每一步 decode 都把**上一步**
最后一层的 hidden 注入第 0 层，让模型踩着自己上一步的终态往前走。

整个 workload 就是一个 `Recurrence` 成员——没有任何任务代码，层号、混合权重、
ramp 全部进签名：

- `source_layer=-1, destination_layer=0`：从最后一层取，注入第 0 层；
- `alpha=1.0, beta=None`：注入位置完全换成（归一化后的）源层 hidden；把 alpha
  调小就是「掺一点」而不是「换掉」，给 beta 一个值则可以单独控制目标层原输出
  的保留比例；
- `ramp_tokens=0`：从第一个 token 起满强度注入；设成 N 则强度在前 N 个 token
  从 0 线性升到满，适合「先正常预热、再逐渐进入循环」。

你可能会嘀咕：这和我自己写个 forward hook 改 hidden 有什么区别？区别在于它进了
签名——引擎知道这一步的输入依赖上一步的终态，逐层的 KV 备份与恢复按声明的
纪律执行，而不是在框架背后悄悄改张量。这也正是它能和本系列其它机制
（状态发布、overlay）严丝合缝组合的原因。

运行后预期看到一段与基线明显不同的续写：第 0 层每步都被注入上一步的终态，
生成轨迹自然分叉。

In [3]:
@dataclass(frozen=True, slots=True)
class StepLoopWorkload(pllm.Workload):
    name = "step_loop"
    config: pllm.model.Recurrence = pllm.model.Recurrence(
        source_layer=-1, destination_layer=0, alpha=1.0, beta=None, ramp_tokens=0)


step = pllm.make_llm(workload=StepLoopWorkload(), config=config())
recirculated = text_of(step, PROMPT)
step.close()
cleanup()
print("step loop:", recirculated)
assert recirculated != baseline

INFO 09-10 18:43:09 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 64, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 64, 'max_num_seqs': 1, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_cls': 'pllm.worker.GPUWorker', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 18:43:10 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 18:43:11 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 18:43:11 [model.py:1883] Using max model len 64


INFO 09-10 18:43:11 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])


INFO 09-10 18:43:12 [core.py:121] Initializing a V1 LLM engine (v0.27.1) with config: model='Qwen/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-0.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=64, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collec

... (9 outputs left out)

INFO 09-10 18:43:14 [kv_cache_utils.py:2236] Maximum concurrency for 64 tokens per request: 5088.75x


INFO 09-10 18:43:14 [gpu_worker.py:789] Free memory on device (11.94/23.48 GiB) on startup. Desired GPU memory utilization is (0.2, 4.7 GiB). Actual usage is 0.97 GiB for consumed memory (weights + non-torch), 0.0 GiB for peak activation, and 0.0 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=3844831232` (3.58 GiB) to fit into requested memory, or `--kv-cache-memory=11617073152` (10.82 GiB) to fully utilize gpu memory. Current kv cache memory in use is 3.73 GiB.


INFO 09-10 18:43:14 [core.py:355] init engine (profile, create kv cache, warmup model) took 0.91 s


step loop:  a new study that shows that the use of a new drug called T-2


## 会话循环：带着上一个会话的终态重新生成

教程 3 里，route 键让多个请求共享同一条 lineage。要把「上一个会话的终态」
交给下一个请求，还差两块积木：**carry 字段**和**携带（pllm.carry）**。

- `carry={"carry": ...}` 声明一个不参与优化的普通字段——没有梯度、没有动量槽，
  但它跟随训练发布、有自己的版本；
- 训练输入上的第二个标注 `pllm.carry("session", "carry")` 表示：发布时，把这个
  输入的**最后一行**写进 `carry`；
- memory 的读取是**输入版本**语义：这一步读到的是上一步发布的值，所以跨步传递
  不需要任何额外的同步开销。

串起来就是：请求 A 的每一步都把当前 hidden 写进 `carry`，A 结束时 `carry` 里
存的是 **A 的终态 hidden**；同 route 的请求 B 第一步就能读到它——B 站在 A 的
终点上重新生成。

怎么证明 B 真的读到了，而不是我们的想象？让训练损失依赖 `carry`
（`prediction = h @ proj.T + carry`），这样同一 prompt 的**首步梯度**就会把
`carry` 的内容暴露出来。我们连发三个请求：A 开新 lineage（carry 为零）、B 走
同一 route（carry 是 A 的终态）、C 换一个全新 route（carry 又是零）。预期结果：
A ≠ B，而 A 与 C **逐元素完全相等**——零状态的确定性，一点噪声都没有。

打开 trace 的梯度选择器，就能把这些梯度拿出来逐个检查。

In [4]:
class SessionLoop(pllm.Workload):
    """route 共享的会话状态：proj 参与训练，carry 是跨请求的 carry 字段。"""

    session = pllm.RequestState(
        {"proj": lambda m: (m.hidden_size, m.hidden_size)},
        optimizer=pllm.AdamW(lr=1e-3),
        carry={"carry": lambda m: (m.hidden_size,)},
        keys=(pllm.state.RequestKeyField.ROUTE,))

    def ready(self, versions: Annotated[torch.Tensor, pllm.StateVersion("session")]) -> torch.Tensor:
        return versions > 0

    @pllm.on(pllm.Phase.BEFORE_SAMPLE, rows="ready")
    @pllm.guide_sampling(beta=0.2)
    def predict(self, h: Annotated[torch.Tensor, pllm.Hidden(-1)],
                proj: Annotated[torch.Tensor, pllm.Parameter("session", "proj")],
                carry: Annotated[torch.Tensor, pllm.Parameter("session", "carry")],
                versions: Annotated[torch.Tensor, pllm.StateVersion("session")]) -> torch.Tensor:
        return torch.bmm(h.unsqueeze(1), proj).squeeze(1) + carry

    @pllm.on(pllm.Phase.AFTER_SAMPLE)
    @pllm.train(state="session", reduction="state_mean")
    def learn(self,
              h: Annotated[torch.Tensor, pllm.Hidden(-1), pllm.carry("session", "carry")],
              proj: Annotated[torch.Tensor, pllm.Parameter("session", "proj")],
              carry: Annotated[torch.Tensor, pllm.Parameter("session", "carry")]) -> torch.Tensor:
        prediction = torch.bmm(h.unsqueeze(1), proj).squeeze(1) + carry
        losses = (prediction - h).square().mean(1)
        return losses  # 每行一个损失，pllm 按状态求均值


# route lineage 在请求结束后休眠保留，需要额外的 lineage 容量
session = pllm.make_llm(workload=SessionLoop(), config=config(max_num_seqs=2))

INFO 09-10 18:43:15 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 64, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 64, 'max_num_seqs': 2, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_cls': 'pllm.worker.GPUWorker', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 18:43:16 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 18:43:16 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 18:43:16 [model.py:1883] Using max model len 64


INFO 09-10 18:43:16 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])


INFO 09-10 18:43:18 [core.py:121] Initializing a V1 LLM engine (v0.27.1) with config: model='Qwen/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-0.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=64, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collec

... (11 outputs left out)

WARNING 09-10 18:43:20 [jit_monitor.py:135] Triton kernel JIT compilation during inference: _prepare_prefill_inputs_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


WARNING 09-10 18:43:20 [jit_monitor.py:135] Triton kernel JIT compilation during inference: _prepare_pos_seq_lens_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


WARNING 09-10 18:43:20 [jit_monitor.py:135] Triton kernel JIT compilation during inference: _gather_block_tables_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


INFO 09-10 18:43:20 [core.py:355] init engine (profile, create kv cache, warmup model) took 0.92 s


In [5]:
from pllm.execution.trace import TraceSelection, TraceEventKind

graph = session.attachment.plan.semantic_graph
learn_index = next(i for i, task in enumerate(graph.tasks) if str(task.symbol) == "learn")
learn_id = next(task.task_id for task in session.attachment.plan.schedule_physical.tasks
                if task.task_id == learn_index)


def first_gradient(prompt, route):
    """该次 generate 新增的首个 learn 梯度（trace 是会话级有界缓冲，用游标切片）。"""
    before = session.trace_snapshot()
    cursor = before.dropped + len(before.events)
    session.select_trace(TraceSelection(tasks=(learn_id,), gradients=True, max_tensor_bytes=32768))
    try:
        session.generate([prompt], SamplingParams(max_tokens=4, temperature=0.0,
            ignore_eos=True), use_tqdm=False, routes=(route,))
        snapshot = session.trace_snapshot()
        events = [e for e in snapshot.events[max(0, cursor - snapshot.dropped):]
                  if e.kind is TraceEventKind.GRADIENT]
        return events[0].tensor() if events else None
    finally:
        session.select_trace(TraceSelection())


grad_a = first_gradient(PROMPT, "s")      # 新 lineage：carry 为零
grad_b = first_gradient(PROMPT, "s")      # 加入 A 的 lineage：carry = A 的终态 hidden
grad_c = first_gradient(PROMPT, "fresh")  # 新 route：又是零

print("首步梯度范数 A/B/C:",
      grad_a.norm().item(), grad_b.norm().item(), grad_c.norm().item())
assert not torch.allclose(grad_a, grad_b), "同一路由的重新生成读到了存储的终态"
assert torch.allclose(grad_a, grad_c), "新路由从零 carry 开始"
session.close()

WARNING 09-10 18:43:20 [jit_monitor.py:135] Triton kernel JIT compilation during inference: _bmm_outer_product_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


首步梯度范数 A/B/C: 0.3170323371887207 0.16873608529567719 0.3170323371887207


## 小结

这一篇我们给引擎装上了三种粒度的「再来一遍」：

- **层循环** = overlay + `layer_route` + `layers.apply_updates()`：更新当步生效；replay 层
  必须是 targets 的子集，未重放的层保留正常前向 KV——这是声明过的近似。
- **步循环** = 一个 `Recurrence` 成员，零任务代码，每步把上一步的终态注入指定层。
- **会话循环** = route 共享 lineage + `pllm.carry` 写 carry 字段；跨请求传递的是
  **状态版本**，不引入任何请求侧历史对象。
- 顺带学会的观察技巧：trace 游标 `dropped + len(events)` 同样适用于跨多次
  generate 的梯度/损失切片。

如果只能记住一句话，就记这句：**循环是声明出来的作用域，不是写出来的控制流。**

下一篇：[教程 5：decode 时在线训练](05_decode_training.ipynb)——把循环再拧紧
一圈：训练目标跨步、跨请求地累积，decode 本身就是训练过程。